In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1. Dataset Overview

In [ ]:
import sys; sys.path.insert(0, '..')
from src.features import load_raw, load_stores, load_items, load_oil, load_holidays

train_raw, test_raw = load_raw()
stores = load_stores()
items  = load_items()
oil    = load_oil()
holidays = load_holidays()

print(f'Train shape:  {train_raw.shape}')
print(f'Test shape:   {test_raw.shape}')
print(f'Date range:   {train_raw.date.min().date()} to {train_raw.date.max().date()}')
print(f'Stores:       {train_raw.store_nbr.nunique()}')
print(f'Items:        {train_raw.item_nbr.nunique()}')
print(f'Unique pairs: {train_raw.groupby(["store_nbr","item_nbr"]).ngroups:,}')

In [ ]:
# Daily sales volume
daily = train_raw.groupby('date')['unit_sales'].sum().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Total daily sales
axes[0,0].plot(daily['date'], daily['unit_sales'], lw=0.8, color='steelblue')
axes[0,0].set_title('Total Daily Sales')
axes[0,0].set_xlabel('Date'); axes[0,0].set_ylabel('Unit Sales')

# 2. Sales distribution (log scale)
log_sales = np.log1p(train_raw['unit_sales'])
axes[0,1].hist(log_sales, bins=80, color='coral', edgecolor='white')
axes[0,1].set_title('Distribution of log1p(unit_sales)')
axes[0,1].set_xlabel('log1p(unit_sales)')

# 3. Oil price over time
axes[1,0].plot(oil['date'], oil['dcoilwtico'], lw=1, color='goldenrod')
axes[1,0].set_title('Ecuador Oil Price (WTI)')
axes[1,0].set_xlabel('Date'); axes[1,0].set_ylabel('USD/barrel')

# 4. Sales by store type
merged_type = train_raw.merge(stores[['store_nbr','type']], on='store_nbr')
type_sales = merged_type.groupby('type')['unit_sales'].mean().sort_values(ascending=False)
axes[1,1].bar(type_sales.index, type_sales.values, color='mediumpurple')
axes[1,1].set_title('Mean Daily Sales by Store Type')
axes[1,1].set_xlabel('Store Type'); axes[1,1].set_ylabel('Mean Unit Sales')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_eda_overview.png'), dpi=150)
plt.show()

In [ ]:
# Weekly seasonality
train_raw['dow'] = train_raw['date'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_sales = train_raw.groupby('dow')['unit_sales'].mean().reindex(dow_order)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(dow_sales.index, dow_sales.values, color='teal')
ax.set_title('Average Unit Sales by Day of Week')
ax.set_ylabel('Mean Unit Sales')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fig_weekly_seasonality.png'), dpi=150)
plt.show()

## 2. Model Comparison

## 3. Feature Importance (CatBoost)

## 4. Error Analysis

## 5. Summary Table